# Rust 124: Advanced String Manipulation

Chapter 24 turns text into **data**. Chapter 4 handled strings as strings — slicing, formatting, comparing. This chapter is the one where Python programmers reach for libraries: `re` for patterns, `csv` for tabular text, `json` for structured payloads, `pandas` for DataFrames, `BeautifulSoup` for HTML. Rust's versions are **`regex`**, **`csv`**, **`serde_json`** (with `serde` derive), **`polars`**, and **`scraper`** — and each one is honest about the same bargain: text stays text until you *declare* its structure, and the parse either produces a typed value or an error you can name.

Everything here is a **runnable example**. Read the markdown, run the cell below it, and change nothing — the practice twin of this notebook is `24_advanced-string-manipulation_p.ipynb`.

## Who this is written for

You are assumed to be a competent programmer arriving from Python. `re.findall`, `csv.DictReader`, `json.loads`, `pandas.read_csv` and `BeautifulSoup(html, "html.parser")` are old friends. What is new in Rust: every parser is a **`Result` with a classified error**, decoded rows land in **structs you declare** (derive, not duck typing), a `DataFrame` has a **schema that is part of the value**, and an HTML tree is a typed **DOM you query with CSS selectors**. The strictness pays once, at the boundary: after the parse, everything downstream compiles against real types.

## How to read this notebook

- Markdown cells explain; code cells demonstrate. **Run each code cell as you reach it.** No files are written anywhere: every parse below runs on string literals and in-memory buffers, so restart-and-run-from-the-top always works.
- Code that intentionally *fails* is shown as a comment, together with the diagnostic you would get. §10 is the catalogue.
- The running domain is the trading desk from the capstone: blotter fills (`sym,qty,px`), a price-tick wire format, an execution-report JSON, a fills table as a DataFrame, and an HTML blotter page.
- One honesty note runs through the chapter: **the formats are invented to be small enough to read**, not because real formats differ in kind. A 4 GB CSV parses with the same `csv::Reader`; a real web page is the same `Selector` calls on a bigger tree.

## The one idea to internalise

Text is **ambiguous** — `"228.60"` is a string, a number, or a column of numbers depending on who is asking. Python's parsers return dictionaries and duck-typed objects and let the ambiguity flow downstream. Rust's parsers end it at the door: the regex gives you *named captures*, the CSV reader gives you *structs via serde*, the JSON gives you either a dynamic `Value` or your struct, the DataFrame carries its *schema*, and the DOM hands you *elements you query by selector*. Choose the representation first (§9); the crates are interchangeable once you do.

## Sections

- §1 regex I: match and capture — `is_match`, `find`, named `captures`, `$name` expansion
- §2 regex II: the workhorses — `split`, `replace_all`, `RegexSet`, flags `(?i)`, `(?m)`, the ownership rule
- §3 csv I: rows as records — `Reader`, `headers`, `records`, `StringRecord`, positions
- §4 csv II: rows as structures — `HashMap` rows, serde `deserialize()` into structs, `Writer`
- §5 JSON I: the dynamic `Value` — indexing that never panics, `as_*` extractors, classified errors
- §6 JSON II: structures both ways — derive roundtrip, `to_value`/`from_value` bridges
- §7 polars: DataFrames from CSV — schema, column selection, filtering, grouping
- §8 scraper: HTML as a queryable DOM — `Html`, `Selector`, text and attribute extraction
- §9 Choosing representations — the chapter's exam, as a table
- §10 E-codes, errors, and kernel traps — including the silent `<tr>` drop
- §11 Mental model: the receipts — the probe results every claim above stands on
- §12 Cheat sheet — all five crates on one page
- §13 What comes next — 25 (tests), 26 (macros)

## Prerequisites

Chapters 4 (strings), 6 (collections), 10 (structs/traits), 11 (errors), 12 (iterators). Chapter 15's polars section is re-used but re-taught from the CSV angle — nothing there is assumed beyond "a DataFrame has columns".


### Contents

- [0. Notebook Conventions: Text In, Types Out](#0-notebook-conventions-text-in-types-out)
- [1. regex I: Match and Capture](#1-regex-i-match-and-capture)
- [2. regex II: Split, Replace, Sets, Flags](#2-regex-ii-split-replace-sets-flags)
- [3. csv I: Rows as Records](#3-csv-i-rows-as-records)
- [4. csv II: Rows as Structures](#4-csv-ii-rows-as-structures)
- [5. JSON I: The Dynamic `Value`](#5-json-i-the-dynamic-value)
- [6. JSON II: Structures Both Ways](#6-json-ii-structures-both-ways)
- [7. polars: DataFrames from CSV](#7-polars-dataframes-from-csv)
- [8. scraper: HTML as a Queryable DOM](#8-scraper-html-as-a-queryable-dom)
- [9. Choosing Representations](#9-choosing-representations)
- [10. E-Codes, Errors, and Kernel Traps](#10-e-codes-errors-and-kernel-traps)
- [11. Mental Model: The Receipts](#11-mental-model-the-receipts)
- [12. Cheat Sheet](#12-cheat-sheet)
- [13. Review](#13-review)

If you are returning to this chapter later, §12 is the one-page summary, §10 the trap catalogue, and §11 the probe receipts behind every behavioural claim below.


## 0. Notebook Conventions: Text In, Types Out

This notebook runs on the **evcxr** kernel, which wraps each cell into a generated program and compiles it incrementally. The standing rules from chapters 17 and 23 all hold: no `fn main`, state persists across cells, items are order-independent, one namespace per notebook, first cell that needs a name does the `use`. Four chapter-specific conventions:

- **Everything parses in memory.** Input text lives in string literals (or `Cursor`s over them); nothing is read from or written to disk. The parse is the point, not the I/O.
- **One crate cell, all five crates.** §0.1 loads `regex`, `csv`, `serde`+`serde_json`, `scraper`, and `polars` once per kernel session; the compile takes minutes **once** (polars is the whale) and is cached afterwards.
- **Derive structs are per-cell.** `#[derive(Serialize, Deserialize)]` structs declared inside a `{}` block stay scoped to that cell — the kernel's one-namespace rule makes top-level redefinition an error, so each section names its demo structs locally (§6 explains the trade).
- **Borrowed text does not persist.** Captures are `&str` slices *of the haystack*; anything you bind at cell top level ends in `.to_string()`/`.into_owned()`/`.to_owned()` or lives inside a block (§2.4, §10).

### The crates

`regex` 1.13 owns §1–2 (match, capture, replace); `csv` 1.4 owns §3–4 (records, serde rows, writer); `serde_json` 1.0 with `serde` derive owns §5–6; `polars` 0.55 (features `lazy, csv, temporal, strings` — the same pin as chapter 15) owns §7; `scraper` 0.27 owns §8. Everything else is `std`.


In [ ]:
// 0.1 The chapter's crate cell: run once per kernel session, features included.
:dep regex = { version = "1" }
:dep csv = { version = "1" }
:dep serde = { version = "1", features = ["derive"] }
:dep serde_json = { version = "1" }
:dep polars = { version = "0.55", features = ["lazy", "csv", "temporal", "strings"] }
:dep scraper = { version = "0" }


In [ ]:
// 0.2 Smoke test: all five crates resolve. If this cell compiles, the session is armed.
let _ = regex::Regex::new("x").unwrap();
let _ = csv::Writer::from_writer(vec![]);
let _ = serde_json::Value::Null;
let _ = polars::prelude::DataType::String;
let _ = scraper::Html::new_fragment();
println!("crates loaded: regex, csv, serde/serde_json, polars, scraper");


## 1. regex I: Match and Capture

Python's `re.search(r"(\w+)-(\w+)", s).group(1)` compiles the pattern *every call* and addresses groups by position. Rust's `regex` crate compiles once into a **`Regex` value**, and the moment your pattern grows past two groups you name them: `(?<sym>...)` and address them as `caps["sym"]`. The crate has no backtracking — it is a finite automaton, linear in the haystack — which is why there is no lookbehind and why catastrophic backtracking cannot happen.

The three questions you ask of a regex, in increasing strength:

1. **Is it there?** — `re.is_match(hay)` → `bool`.
2. **Where is it?** — `re.find(hay)` → `Option<Match>` with `.start()`/`.end()`/`.as_str()`.
3. **What are the parts?** — `re.captures(hay)` → `Option<Captures>`, indexed by name or number.


In [ ]:
// 1.1 The desk's tick format: SYM-SIDE@PRICE. Compile once, name the parts.
{
    use regex::Regex;
    let tick = Regex::new(r"(?<sym>[A-Z]{3,4})-(?<side>BUY|SELL)@(?<px>[0-9]+\.[0-9]{2})").unwrap();

    let line = "AAPL-BUY@228.60 MSFT-SELL@418.12";

    // (1) is it there?
    assert!(tick.is_match(line));
    // (2) where is the first one?
    let m = tick.find(line).unwrap();
    assert_eq!((m.start(), m.end()), (0, 15));
    // (3) what are the parts?
    let caps = tick.captures(line).unwrap();
    assert_eq!(&caps["sym"], "AAPL");
    assert_eq!(&caps["side"], "BUY");
    assert_eq!(&caps["px"], "228.60");
    // groups: the whole match is group 0, so three named groups make len() 4
    assert_eq!(caps.len(), 4);

    println!("first tick: {} {} @ {}", &caps["side"], &caps["sym"], &caps["px"]);
}


In [ ]:
// 1.2 No match is a value, not an exception: `None` means "not there", and the
// error case you should fear is the *pattern* failing to compile — Regex::new
// returns Result precisely so the parse of your pattern is checked once.
{
    use regex::Regex;
    let tick = Regex::new(r"(?<sym>[A-Z]{3,4})-(?<side>BUY|SELL)@(?<px>[0-9]+\.[0-9]{2})").unwrap();

    let quiet = "no ticks here";
    assert!(tick.captures(quiet).is_none());
    assert_eq!(tick.find(quiet).map(|m| m.as_str()), None);

    // a pattern that cannot compile is a programmer error, surfaced at compile-time-of-the-pattern:
    let broken = Regex::new(r"(?<sym>[A-Z");
    assert!(broken.is_err());

    println!("absence is None; bad patterns are Err at Regex::new");
}


`captures` finds the **first** match only. For every match in the haystack, `captures_iter` hands you each `Captures` in turn — the analogue of `re.finditer`. And the capture *names* are data too: `capture_names()` lists them (with `None` for unnamed groups), which is how you can write tooling that reports what a pattern extracts.


In [ ]:
// 1.3 All matches: captures_iter, the finditer analogue. Note what the captures
// borrow: `&str` slices of the haystack, valid as long as it lives.
{
    use regex::Regex;
    let tick = Regex::new(r"(?<sym>[A-Z]{3,4})-(?<side>BUY|SELL)@(?<px>[0-9]+\.[0-9]{2})").unwrap();

    let line = "AAPL-BUY@228.60 MSFT-SELL@418.12 TSLA-BUY@250.00";
    let parsed: Vec<(String, String)> = tick
        .captures_iter(line)
        .map(|c| (c["sym"].to_string(), c["px"].to_string()))
        .collect();
    assert_eq!(parsed.len(), 3);
    assert_eq!(parsed[2], ("TSLA".to_string(), "250.00".to_string()));

    // the pattern's own names, as data:
    let names: Vec<Option<&str>> = tick.capture_names().collect();
    assert_eq!(names, vec![None, Some("sym"), Some("side"), Some("px")]);

    println!("ticks: {parsed:?}");
}


## 2. regex II: Split, Replace, Sets, Flags

Python's `re.split`, `re.sub`, and inline flags all have direct analogues — with one departure worth naming early: **`replace_all` accepts expansion syntax, not closures** (`$1`, `$sym`, `${name}`), and the result is a `Cow<str>` — borrowed when nothing changed, owned when the replacement altered the text. The chapter's ownership rule (§0) applies: `.into_owned()` before you bind it at top level.

Also new vs Python: **`RegexSet`** — compile *many* patterns once and test which ones match, the right tool for "which of these line categories is this?" dispatch.


In [ ]:
// 2.1 replace_all with $name expansion, and what Cow means for you.
{
    use regex::Regex;
    let tick = Regex::new(r"(?<sym>[A-Z]{3,4})-(?<side>BUY|SELL)@(?<px>[0-9]+\.[0-9]{2})").unwrap();

    let line = "AAPL-BUY@228.60 MSFT-SELL@418.12";
    // "$side $sym @ $px" reorders the fields — no closure needed:
    let rebased = tick.replace_all(line, "$side $sym @ $px");
    assert_eq!(rebased, "BUY AAPL @ 228.60 SELL MSFT @ 418.12");

    // $0 is the whole match; ${name} disambiguates from following text:
    let tagged = tick.replace_all(line, "[$0]");
    assert_eq!(tagged, "[AAPL-BUY@228.60] [MSFT-SELL@418.12]");

    // a pattern that matches NOTHING returns the haystack borrowed, not a copy:
    let untouched = tick.replace_all("plain text", "X");
    assert_eq!(untouched, "plain text");

    // the one rule: own it before it outlives the haystack
    let owned: String = rebased.into_owned();
    println!("{owned}");
}


In [ ]:
// 2.2 split on a compiled separator; the empty-item rule matches Python's.
{
    use regex::Regex;
    let sep = Regex::new(r"[,;]\s*").unwrap();

    let fields: Vec<&str> = sep.split("a, b;c").collect();
    assert_eq!(fields, vec!["a", "b", "c"]);

    // Python rule preserved: a separator at either end yields an empty item
    let ragged: Vec<&str> = sep.split(",a,").collect();
    assert_eq!(ragged, vec!["", "a", ""]);
    println!("split: {fields:?}");
}


In [ ]:
// 2.3 RegexSet: which of several patterns match, compiled once.
{
    use regex::RegexSet;
    let categories = RegexSet::new([
        r"BUY@",
        r"@[0-9]{3}\.",
        r"TSLA",
    ]).unwrap();

    let line = "AAPL-BUY@228.60";
    let hits = categories.matches(line);
    assert!(hits.matched(0) && hits.matched(1) && !hits.matched(2));
    println!("matched categories: {:?}", hits.into_iter().collect::<Vec<_>>());
}


In [ ]:
// 2.4 Flags are inline, as in Python: (?i) ignore case, (?m) multi-line anchors,
// (?s) dot-matches-newline. This scan classifies a desk log's lines.
{
    use regex::Regex;
    let warn = Regex::new(r"(?im)^\[(warn|error)\]\s+(.+)$").unwrap();

    let log = "[warn] slow fill\n[error] rejected: bad px\n[info] ok\n[WARN] retrying";
    let msgs: Vec<String> = warn
        .captures_iter(log)
        .map(|c| c[2].to_string())
        .collect();
    assert_eq!(msgs.len(), 3);
    assert_eq!(msgs[0], "slow fill");
    println!("flagged lines: {msgs:?}");
}


### 2.4 The ownership rule, once more

Every capture is a `&str` **into the haystack**. Delete the `.to_string()` in §2.4's map and the code still compiles — *in that cell*, because `log` outlives the loop. Move the collection to cell top level and evcxr will eventually hand you E0597 (`borrowed value does not live long enough`) or a persisted-`Cow` error the moment the haystack is cell-local. The habit that survives the kernel: **parse inside a block, persist owned values** (`String`, numbers, your structs).

```rust
// let all: Vec<Captures> = warn.captures_iter(log).collect(); // if `log` is
// cell-local and `all` is top-level, the borrow outlives the haystack — E0597.
```


## 3. csv I: Rows as Records

Python's `csv.reader` yields lists of strings; `csv.DictReader` zips them with the header. The `csv` crate splits these the same way — and fixes one Python wart: **it parses the CSV grammar properly** (quoted fields, embedded commas, `""` escapes) where `str::split(',')` quietly corrupts every row containing `", "`. §3 stays at the record level: headers, rows, and the position metadata that makes debugging a 4 GB feed possible.

The cast:

- **`Reader`** — created `from_reader(any Read)`; a `&str` is a `Read` of bytes via `as_bytes()`.
- **`StringRecord`** — one row: `get(i)` by column index, `len()`, `iter()`.
- **`headers()`** — the first row as a `StringRecord`, when `has_headers` is on (the default).
- **`.position()`** — line/byte offsets into the source, on both records and errors.


In [ ]:
// 3.1 The reader loop: headers + records. The blotter CSV has a quoted comma —
// split(',') would butcher row 2; the crate's parser handles it.
{
    use std::io::Cursor;

    const BLOTTER: &str = "sym,qty,px\nAAPL,100,228.60\n\"MS,FT\",250,418.12\n";

    let mut rdr = csv::Reader::from_reader(Cursor::new(BLOTTER));
    let headers = rdr.headers().unwrap().clone();

    let mut recs: Vec<csv::StringRecord> = Vec::new();
    for result in rdr.records() {
        recs.push(result.unwrap());
    }

    let header_names: Vec<String> = headers.iter().map(String::from).collect();
    assert_eq!(header_names, ["sym", "qty", "px"]);
    assert_eq!(recs.len(), 2);
    assert_eq!(recs[0].get(0), Some("AAPL"));
    assert_eq!(recs[0].get(2), Some("228.60"));
    // the quoted comma survives as ONE field:
    assert_eq!(recs[1].get(0), Some("MS,FT"));
    assert_eq!(recs[1].len(), 3);

    println!("headers = {:?}, row 1 = {:?}", header_names, recs[1].iter().collect::<Vec<_>>());
}


In [ ]:
// 3.2 Position metadata: every record knows where it came from — the line
// number in the ORIGINAL source (headers are line 1). This is how you point
// a human at row 2 of a feed that failed.
{
    use std::io::Cursor;

    const BLOTTER: &str = "sym,qty,px\nAAPL,100,228.60\n\"MS,FT\",250,418.12\n";

    let mut rdr = csv::Reader::from_reader(Cursor::new(BLOTTER));
    let mut lines = Vec::new();
    while let Some(r) = rdr.records().next() {
        let rec = r.unwrap();
        let pos = rec.position().unwrap();
        lines.push((pos.line(), rec.get(0).unwrap_or("?").to_string()));
    }
    assert_eq!(lines, vec![(2, "AAPL".to_string()), (3, "MS,FT".to_string())]);
    println!("record -> source line: {lines:?}");
}


In [ ]:
// 3.3 Malformed rows are RESULTS, not panics: a row with the wrong field count
// is an UnequalLengths error you can inspect — and `flexible(true)` turns the
// check off when ragged feeds are a fact of life.
{
    use std::io::Cursor;

    const RAGGED: &str = "sym,qty,px\nAAPL,100,228.60\nMSFT,250\n";

    // strict (default): the second row is an Err with the row number attached
    let mut strict = csv::Reader::from_reader(Cursor::new(RAGGED));
    let mut got_err = None;
    for result in strict.records() {
        if result.is_err() {
            got_err = Some(result.unwrap_err());
            break;
        }
    }
    let e = got_err.unwrap();
    println!("strict: {:?}", e.kind());

    // flexible: the same feed reads through
    let mut loose = csv::ReaderBuilder::new().flexible(true).from_reader(Cursor::new(RAGGED));
    let n = loose.records().map(|r| r.unwrap()).count();
    assert_eq!(n, 2);
    println!("flexible: both rows read");
}


## 4. csv II: Rows as Structures

`Vec<StringRecord>` is still "text, indexed by number". The chapter's promise — *structure* — happens in two steps up:

1. **`HashMap<String, String>` rows** — zip the headers with each record (the `DictReader` analogue, built by hand so you see the seam).
2. **serde `deserialize()`** — declare a struct with `#[derive(Deserialize)]` and the reader yields **typed rows directly**: `String`/`i64`/`f64` fields, parse errors surfaced per-row. This is the crate's headline feature and the pattern the whole series has been driving toward: text in, structs out, nothing stringly-typed downstream.

Writing goes the same way: a `Writer` accepts slices, and quotes any field that needs it.


In [ ]:
// 4.1 The DictReader analogue, built by hand: zip headers with values.
// `has_headers(false)` turns the first row into DATA so it can be the zip key.
{
    use std::collections::HashMap;
    use std::io::Cursor;

    const BLOTTER: &str = "sym,qty,px\nAAPL,100,228.60\nMSFT,250,418.12\n";

    let mut rdr = csv::ReaderBuilder::new().has_headers(false).from_reader(Cursor::new(BLOTTER));
    let mut iter = rdr.records();

    let headers: Vec<String> = iter.next().unwrap().unwrap().iter().map(String::from).collect();
    let rows: Vec<HashMap<String, String>> = iter
        .map(|r| {
            headers
                .iter()
                .cloned()
                .zip(r.unwrap().iter().map(String::from))
                .collect()
        })
        .collect();

    assert_eq!(rows.len(), 2);
    assert_eq!(rows[0]["sym"], "AAPL");
    assert_eq!(rows[1]["px"], "418.12");
    println!("row 0 = {:?}", rows[0]);
}


In [ ]:
// 4.2 The headline: Reader::deserialize() yields typed structs. With headers
// on (the default), CSV columns match struct fields BY NAME.
{
    use serde::Deserialize;
    use std::io::Cursor;

    #[derive(Debug, Deserialize)]
    struct Fill {
        sym: String,
        qty: i64,
        px: f64,
    }

    const BLOTTER: &str = "sym,qty,px\nAAPL,100,228.60\n\"MS,FT\",250,418.12\n";
    let mut rdr = csv::Reader::from_reader(Cursor::new(BLOTTER));
    let fills: Vec<Fill> = rdr.deserialize().map(|r| r.unwrap()).collect();

    assert_eq!(fills.len(), 2);
    assert_eq!(fills[0].sym, "AAPL");
    assert_eq!(fills[0].qty, 100);
    assert!((fills[0].px - 228.60).abs() < 1e-9);
    assert_eq!(fills[1].sym, "MS,FT");
    println!("fills = {fills:?}");
}


In [ ]:
// 4.3 Bad cells are per-row Errs: "12x" is not an i64. The deserialize stream
// hands you the Result — skip it, fail it, or collect it; the choice is yours,
// which is exactly what a silently-corrupting parser would have hidden.
{
    use serde::Deserialize;
    use std::io::Cursor;

    #[derive(Debug, Deserialize)]
    struct Fill {
        sym: String,
        qty: i64,
    }

    const DIRTY: &str = "sym,qty\nAAPL,100\nMSFT,12x\nTSLA,250\n";
    let mut rdr = csv::Reader::from_reader(Cursor::new(DIRTY));

    let mut good: Vec<Fill> = Vec::new(); // annotate: deserialize() is generic over the row type
    let mut bad_rows = 0;
    for row in rdr.deserialize() {
        match row {
            Ok(f) => good.push(f),
            Err(_) => bad_rows += 1,
        }
    }
    assert_eq!(good.len(), 2);
    assert_eq!(good[0].qty, 100); // read the fields, or dead-code warnings fire
    assert_eq!(bad_rows, 1);
    assert_eq!(good[1].sym, "TSLA");
    println!("good = {good:?}, rejected {bad_rows} row");
}


In [ ]:
// 4.4 The writer: slices in, RFC-4180 text out. A field containing a comma is
// quoted on write — the exact inverse of §3.1's read.
{
    let mut w = csv::Writer::from_writer(vec![]);
    w.write_record(["sym", "qty"]).unwrap();
    w.write_record(["AAPL", "100"]).unwrap();
    w.write_record(["MS,FT", "250"]).unwrap();

    let bytes = w.into_inner().unwrap();
    let text = String::from_utf8(bytes).unwrap();
    assert_eq!(text, "sym,qty\nAAPL,100\n\"MS,FT\",250\n");
    println!("wrote:\n{text}");
}


## 5. JSON I: The Dynamic `Value`

Python's `json.loads` returns dicts/lists you index freely — and learn about shape errors at 3 a.m. Rust's dynamic twin is **`serde_json::Value`**: an enum (`Null`, `Bool`, `Number`, `String`, `Array`, `Object`) whose indexing **never panics** — `v["missing"]` is `Value::Null`, `[999]` out of bounds is `Null`. Extractors (`as_str`, `as_i64`, `as_f64`, `as_bool`, `as_array`) return `Option`s, so "is this actually a number?" is a question you answer, not a crash you defer.

When the text is not JSON at all, `from_str` returns a **classified error**: `Syntax`/`Io`/`Data`/`Eof`, with line and column. That classification (not the message string) is what you match on. One kernel-scope note: the `use serde_json::{Map, Value};` below sits at **items level** — the gate concatenates every cell's scope, so a second block-level `use Value` in §5.2 would collide (E0252).


In [ ]:
// 5.1 Dynamic access on the desk's execution report. Indexing never panics;
// missing keys are Null; extractors are Options.
use serde_json::{Map, Value};

{
    let report = r#"{
        "desk": "md-cache",
        "fills": [{"sym": "AAPL", "qty": 100, "px": 228.60}],
        "open": true,
        "note": null
    }"#;
    let v: Value = serde_json::from_str(report).unwrap();

    assert_eq!(v["desk"], "md-cache");
    assert_eq!(v["fills"][0]["qty"], 100);
    assert_eq!(v["fills"][0]["px"].as_f64(), Some(228.60));
    assert_eq!(v["open"].as_bool(), Some(true));
    assert!(v["note"].is_null());
    // the two panics Python would have had, as values:
    assert!(v["no_such_key"].is_null());
    assert!(v["fills"][7]["qty"].is_null());

    println!("desk = {}, first fill px = {}", v["desk"], v["fills"][0]["px"]);
}


In [ ]:
// 5.2 Type mismatches are None (or Err), not TypeErrors: "228.60" is a STRING
// here, so as_f64 is None — you check, then convert.
{
    let v: Value = serde_json::from_str(r#"{"px": "228.60", "qty": 100}"#).unwrap();

    assert_eq!(v["px"].as_f64(), None); // it is a JSON string, not a number
    assert_eq!(v["px"].as_str(), Some("228.60"));
    let px: f64 = v["px"].as_str().unwrap().parse().unwrap(); // explicit, on your terms
    assert_eq!(v["qty"].as_i64(), Some(100));

    // building dynamic values by hand (Map/Array/the From impls):
    let mut m = Map::new();
    m.insert("sym".into(), Value::String("AAPL".into()));
    m.insert("qty".into(), Value::from(100));
    let built = Value::Object(m);
    assert_eq!(built.to_string(), r#"{"qty":100,"sym":"AAPL"}"#); // keys sorted

    println!("string px parsed = {px}; built = {built}");
}


In [ ]:
// 5.3 Malformed JSON: a classified error with coordinates. Match the CATEGORY,
// not the message: Syntax for "not JSON", Data for "valid JSON, wrong shape".
{
    let bad = r#"{"sym": "AAPL", "qty": tru}"#;
    let e = serde_json::from_str::<serde_json::Value>(bad).unwrap_err();
    println!("{:?} @ line {} col {}", e.classify(), e.line(), e.column());

    // "valid JSON, not the shape we need" is the Data class:
    let shape = serde_json::from_str::<i64>(r#""not a number""#).unwrap_err();
    println!("{:?}", shape.classify());
}


## 6. JSON II: Structures Both Ways

The dynamic `Value` is for *exploring*. When the shape is known — and it usually is — you declare it: `#[derive(Serialize, Deserialize)]` on a struct, and `serde_json` converts **both directions**: `to_string`/`from_str` (JSON text), `to_value`/`from_value` (the dynamic `Value` bridge, for when one library hands you `Value` and another wants your struct).

This is the Python `dataclass` + `json` dance, minus the manual field-by-field copying — and the roundtrip is checked by the compiler, not by your test suite's optimism. One kernel-era convention applies: the derive structs in this notebook are **declared inside blocks**, so each section owns its names without colliding across cells (§0).


In [ ]:
// 6.1 The full roundtrip: struct -> text -> struct. Field names become JSON
// keys; the types carry the parse.
{
    use serde::{Deserialize, Serialize};

    #[derive(Debug, Serialize, Deserialize, PartialEq)]
    struct Fill {
        sym: String,
        qty: i64,
        px: f64,
    }

    let f = Fill { sym: "AAPL".into(), qty: 100, px: 228.60 };

    let text = serde_json::to_string(&f).unwrap();
    assert_eq!(text, r#"{"sym":"AAPL","qty":100,"px":228.6}"#);

    let back: Fill = serde_json::from_str(&text).unwrap();
    assert_eq!(back, f); // PartialEq — the roundtrip is assertable, not eyeballed

    // pretty form for humans:
    let pretty = serde_json::to_string_pretty(&f).unwrap();
    assert!(pretty.contains("\n  \"sym\""));

    println!("{text}");
}


In [ ]:
// 6.2 The Value bridge: to_value (struct -> dynamic) and from_value (dynamic
// -> struct). Use it at the seam between "some library gave me Value" and
// "my code wants typed structs".
{
    use serde::{Deserialize, Serialize};

    #[derive(Debug, Serialize, Deserialize)]
    struct Fill {
        sym: String,
        qty: i64,
    }

    let f = Fill { sym: "MSFT".into(), qty: 250 };
    let v = serde_json::to_value(&f).unwrap();
    assert_eq!(v["sym"], "MSFT"); // still dynamically indexable

    let f2: Fill = serde_json::from_value(v).unwrap();
    assert_eq!(f2.qty, 250);
    println!("bridge ok: {f2:?}");
}


In [ ]:
// 6.3 Vec<Fill> and nested shapes serialize exactly as you'd hope; a wrong
// incoming shape is a typed Err naming the field.
{
    use serde::{Deserialize, Serialize};

    #[derive(Debug, Serialize, Deserialize)]
    struct Fill {
        sym: String,
        qty: i64,
    }

    let blotter = vec![
        Fill { sym: "AAPL".into(), qty: 100 },
        Fill { sym: "TSLA".into(), qty: 40 },
    ];
    let text = serde_json::to_string(&blotter).unwrap();
    let back: Vec<Fill> = serde_json::from_str(&text).unwrap();
    assert_eq!(back.len(), 2);

    // wrong shape -> Err (Data class), message names the offending field:
    let wrong = serde_json::from_str::<Fill>(r#"{"sym":"AAPL","qty":"hundred"}"#).unwrap_err();
    println!("vec ok, wrong-shape: {}", wrong);
}


## 7. polars: DataFrames from CSV

Chapter 15 introduced polars as the pandas analogue; this chapter uses it for the thing you actually do with tabular text: **read a CSV into a typed columnar frame and ask questions of it**. The DataFrame is *text-to-structure* taken to the limit — every column has a schema type (`str`, `i64`, `f64`), and the schema is part of the value, printed right in the frame's header.

Three rules recycled from chapter 15 (§7.1–7.3) and one new:

- `SerReader::finish` is a **trait method** — `use polars::prelude::*` brings it (or import the trait alone).
- `CsvReadOptions` builds the reader; `with_has_header(true)` keeps row 1 as names.
- In-memory sources go through `Cursor` — and **`Cursor` itself needs its `use`** (it is not part of polars' prelude; §10 collects the gate receipts).
- **New:** `lazy()` turns the frame into a query plan — `filter`/`group_by`/`agg`/`collect` — where the whole chain type-checks before any row moves (the pandas `.query()` analogue, compiled).


In [ ]:
// 7.1 CSV text -> DataFrame in memory: Cursor + CsvReadOptions + SerReader.
{
    use polars::prelude::*;
    use std::io::Cursor;

    const BLOTTER: &str = "sym,qty,px\nAAPL,100,228.60\nMSFT,250,418.12\nTSLA,40,250.00\n";

    let df = CsvReadOptions::default()
        .with_has_header(true)
        .into_reader_with_file_handle(Cursor::new(BLOTTER))
        .finish()
        .unwrap();

    assert_eq!(df.shape(), (3, 3));
    assert_eq!(df.height(), 3);
    // the schema is INFERRED per column and is part of the value:
    // sym -> String, qty -> Int64, px -> Float64 (see the frame's header)
    let dtypes: Vec<_> = df.dtypes();
    assert_eq!(dtypes.iter().filter(|d| **d == DataType::Int64).count(), 1);
    assert_eq!(dtypes.iter().filter(|d| **d == DataType::Float64).count(), 1);
    assert_eq!(dtypes.iter().filter(|d| **d == DataType::String).count(), 1);

    println!("{df}");
}


In [ ]:
// 7.2 Column access: by name -> Series; the Series' type decides what you can
// ask of it (i64 sums; f64 means). A column that does not exist is an Err
// naming the column — a PolarsError value, not a panic.
{
    use polars::prelude::*;
    use std::io::Cursor;

    const BLOTTER: &str = "sym,qty,px\nAAPL,100,228.60\nMSFT,250,418.12\nTSLA,40,250.00\n";
    let df = CsvReadOptions::default()
        .with_has_header(true)
        .into_reader_with_file_handle(Cursor::new(BLOTTER))
        .finish()
        .unwrap();

    let qty = df.column("qty").unwrap().as_series().unwrap();
    // into_no_null_iter (not into_iter): the ChunkedArray iterator that skips nulls
    // is the one ch15's gate proved compiles for sums over i64 views
    let total: i64 = qty.i64().unwrap().into_no_null_iter().sum();
    assert_eq!(total, 390);

    let missing = df.column("venue");
    assert!(missing.is_err());

    println!("total qty = {total}, missing col is err: {}", missing.is_err());
}


In [ ]:
// 7.3 The lazy query plan: filter, group_by, aggregate, collect. The chain
// type-checks as a whole; `collect` runs it.
{
    use polars::prelude::*;
    use std::io::Cursor;

    const BLOTTER: &str = "sym,side,qty,px\nAAPL,BUY,100,228.60\nMSFT,SELL,250,418.12\nAAPL,SELL,40,229.10\nTSLA,BUY,40,250.00\n";
    let df = CsvReadOptions::default()
        .with_has_header(true)
        .into_reader_with_file_handle(Cursor::new(BLOTTER))
        .finish()
        .unwrap();

    // buys only, per symbol, total quantity:
    let out = df
        .clone()
        .lazy()
        .filter(col("side").eq(lit("BUY")))
        .group_by([col("sym")])
        .agg([col("qty").sum()])
        .sort(["sym"], SortMultipleOptions::default())
        .collect()
        .unwrap();

    assert_eq!(out.height(), 2); // AAPL and TSLA bought
    println!("{out}");

    // the same frame, one filtered row:
    let big = df
        .lazy()
        .filter(col("qty").gt(lit(50)))
        .collect()
        .unwrap();
    assert_eq!(big.height(), 2);
}


## 8. scraper: HTML as a Queryable DOM

Python's `BeautifulSoup(html, "html.parser")` + `soup.select("tr.fill td")` maps one-to-one: **`Html::parse_document`** (a full page) or **`Html::parse_fragment`** (a snippet) builds the typed DOM, **`Selector::parse`** compiles a CSS selector (an `Result` — invalid selectors are a programmer error, caught early), and **`.select(&sel)`** iterates matching elements. Text comes from `.text()` (an iterator of `&str` pieces — `.collect::<String>()` joins them); attributes from `el.value().attr("name")` (an `Option` — attributes are absent or present, never defaulted).

The chapter's sharpest trap lives here and it is **silent**: the HTML5 grammar *forbids* `<tr>` outside a table context, and the parser **drops such elements without a word** (§8.3, §10). BeautifulSoup's html.parser tolerates it; html5ever (scraper's engine, the same spec browsers implement) does not. When your table query returns nothing, suspect the orphan-`<tr>` rule first.


In [ ]:
// 8.1 parse_document + select: the blotter page, read like BeautifulSoup.
{
    use scraper::{Html, Selector};

    let page = Html::parse_document(
        r#"<html><body>
            <h1>Desk blotter</h1>
            <table id="blotter">
              <tr class="fill"><td>AAPL</td><td>100</td><td>228.60</td></tr>
              <tr class="fill"><td>MSFT</td><td>250</td><td>418.12</td></tr>
            </table>
            <a href="/fills/1">detail</a>
        </body></html>"#,
    );

    let row = Selector::parse("tr.fill td").unwrap();
    let cells: Vec<String> = page.select(&row).map(|e| e.text().collect()).collect();
    assert_eq!(cells, ["AAPL", "100", "228.60", "MSFT", "250", "418.12"]);

    let link = Selector::parse(r#"a[href^="/fills/"]"#).unwrap();
    let href = page.select(&link).next().and_then(|e| e.value().attr("href"));
    assert_eq!(href, Some("/fills/1"));

    println!("cells: {cells:?} | first link: {href:?}");
}


In [ ]:
// 8.2 Selector composition: id (#), class (.), attribute ([..]), descendants
// (space). Selector::parse is checked ONCE — an invalid selector is an Err,
// not a runtime surprise per element.
{
    use scraper::{Html, Selector};

    let frag = Html::parse_fragment(
        r#"<p class="price up">228.60</p><p class="price">418.12</p><div id="v">vol</div>"#,
    );

    let up = Selector::parse("p.price.up").unwrap(); // BOTH classes
    let ups: Vec<String> = frag.select(&up).map(|e| e.text().collect()).collect();
    assert_eq!(ups, ["228.60"]);

    let by_id = Selector::parse("div#v").unwrap();
    assert_eq!(frag.select(&by_id).count(), 1);

    assert!(Selector::parse("a >>").is_err()); // invalid CSS rejected here

    println!("compound classes: {ups:?}");
}


In [ ]:
// 8.3 THE TRAP: a <tr> outside a table is DROPPED by the html5 parser —
// no error, no warning, zero matches. The same markup inside <table>...</table>
// selects fine. BeautifulSoup (html.parser) would have kept the row.
{
    use scraper::{Html, Selector};

    let orphan = Html::parse_fragment(r#"<div><tr class="fill"><td>ORPHAN</td></tr></div>"#);
    let sel = Selector::parse("tr.fill td").unwrap();
    let dropped = orphan.select(&sel).count();
    assert_eq!(dropped, 0);

    let wrapped = Html::parse_fragment(r#"<table><tr class="fill"><td>ORPHAN</td></tr></table>"#);
    assert_eq!(wrapped.select(&sel).count(), 1);

    println!("orphan tr matches = {dropped} (dropped!); inside <table> = 1");
}


The rest of scraper's surface is the same shape: `el.value().name()` for the tag name, `el.children()` to walk the tree manually, `doc.root_element()` as the query root. One more ownership note for the kernel: `Html` owns the tree and `select` borrows it — collect `String`s out of `.text()` before the `Html` leaves scope, exactly like regex captures borrow the haystack (§2.4).


## 9. Choosing Representations

Five crates, five representations of the same blotter. This is the chapter's exam — not "how do I call `captures`" but **"which structure does this problem want?"** — asked the way chapter 15 asked it for `chrono` vs `time` vs `Decimal`.

| You need… | Reach for | You get | You pay |
|---|---|---|---|
| to pull fields out of semi-structured text | **`regex`** `captures` | named `&str` groups | patterns to maintain; no nesting |
| to turn one pattern into many text results | **`regex`** `replace_all`/`split` | `Cow<str>` / iterators | same |
| to answer "which of these patterns?" | **`regex`** `RegexSet` | matched indexes | — |
| rows you can address **by column name**, still strings | **`csv`** + `HashMap` zip | `Vec<HashMap<String,String>>` | all values are `String` |
| rows as **typed structs**, dirty rows rejected | **`csv`** + serde derive | `Vec<Fill>` via `deserialize()` | a struct per shape |
| to explore **unknown shape** JSON | **`serde_json::Value`** | panic-free dynamic access | `as_*`/`Option` plumbing |
| JSON whose shape you **know** | **serde derive** | structs, both directions | a struct per shape |
| **columnar** questions (sum, filter, group) | **`polars`** | `DataFrame` with schema | the polars compile; the API's grammar |
| to read **HTML** for data | **`scraper`** | DOM + CSS selectors | html5 grammar strictness (orphan `<tr>`) |

Two rules cut across every row:

- **Declare the target, then parse into it.** The struct (or `Value`) decides the parse; retrofitting structure after a `HashMap<String,String>` sprint is how stringly-typed systems grow.
- **Parse errors are data.** `None` from `as_f64`, `Err` from `deserialize`, `UnequalLengths` from the reader — every crate surfaces the same idea: the boundary either produces a typed value or a nameable failure, never a silent guess.


In [ ]:
// 9.1 One text, four representations: the same tick through regex, csv
// (HashMap and serde) and serde_json. The target type drives everything.
{
    use regex::Regex;
    use serde::Deserialize;
    use std::collections::HashMap;
    use std::io::Cursor;

    const LINE: &str = "AAPL-BUY@228.60";

    // regex: named captures into the line
    let tick = Regex::new(r"(?<sym>[A-Z]+)-(?<side>BUY|SELL)@(?<px>[0-9.]+)").unwrap();
    let c = tick.captures(LINE).unwrap();
    let (sym, side, px) = (c["sym"].to_string(), c["side"].to_string(), c["px"].to_string());

    // row -> CSV text via the writer (a header row, then the tick as fields)
    let mut wtr = csv::Writer::from_writer(vec![]);
    wtr.write_record(["sym", "side", "px"]).unwrap();
    wtr.write_record([&sym, &side, &px]).unwrap();
    let text = String::from_utf8(wtr.into_inner().unwrap()).unwrap();

    // csv: header-zipped HashMap (has_headers(false): both rows are data)
    let mut rdr = csv::ReaderBuilder::new()
        .has_headers(false)
        .from_reader(Cursor::new(text.clone()));
    let mut it = rdr.records();
    let keys = it.next().unwrap().unwrap();
    let vals = it.next().unwrap().unwrap();
    let row: HashMap<String, String> = keys
        .iter()
        .map(String::from)
        .zip(vals.iter().map(String::from))
        .collect();

    // ... or serde (typed):
    #[derive(Debug, Deserialize)]
    struct Fill {
        sym: String,
        side: String,
        px: String,
    }
    let mut rdr2 = csv::Reader::from_reader(Cursor::new(text.clone()));
    let fill: Fill = rdr2.deserialize().next().unwrap().unwrap();

    // serde_json: the same shape as JSON text
    let json_text = format!(
        r#"{{"sym":"{}","side":"{}","px":"{}"}}"#,
        sym, side, px
    );
    let j: FillJson = serde_json::from_str(&json_text).unwrap();

    assert_eq!(row["sym"], "AAPL");
    assert_eq!(fill.sym, "AAPL");
    assert_eq!(fill.side, "BUY");
    assert_eq!(fill.px, "228.60");
    assert_eq!(j.sym, "AAPL");
    assert_eq!(j.side, "BUY");
    assert_eq!(j.px, "228.60");
    println!("four representations, one row: {fill:?}");
}

// the derive target for the json leg (items level, like every shared name)
use serde::Deserialize;
#[derive(Debug, Deserialize)]
struct FillJson {
    sym: String,
    side: String,
    px: String,
}


## 10. E-Codes, Errors, and Kernel Traps

Chapters 4 and 15 collected the string lessons; this chapter adds the five-crate catalogue. Compile-time diagnostics first, then the **runtime** traps that no compiler can flag — including one that fails *silently*.

| Diagnostic / trap | What it means | The fix this chapter uses |
|---|---|---|
| **E0599** `no method named 'finish'` on `CsvReader` | `finish` lives on the `SerReader` **trait**, not the struct | `use polars::prelude::*` (or import the trait alone) — the ch15 `Datelike` lesson again |
| **E0433** `cannot find type 'Cursor'` under polars cells | `Cursor` is `std::io`'s, not polars' — the prelude does not carry it | `use std::io::Cursor;` next to the polars import (this chapter's own gate receipt) |
| **E0277** `the trait bound 'T: serde::Serialize' is not satisfied` on a *derived* struct | two **crate instances** of serde in one namespace — derive impls registered under one are invisible to the other | one serde per process: the `:dep` cell here, the `_externs_24.txt` wave for the gate — never mixed |
| **E0252** `the name 'Value' is defined multiple times` | the gate concatenates cell scopes; a second block-level `use serde_json::Value` collides with the first | one `use` per name at **items level** (§5.1), blocks consume it |
| **E0597** `borrowed value does not live long enough` | a capture/`Cow` borrow outlived the haystack at cell top level | parse in a block; persist owned values (`String`, `.into_owned()`, your structs) |
| **E0433/E0432** unresolved crate/import | the crate cell never ran (crates are per-session) | run §0.1 first; restart-and-run-from-the-top always works |
| **`None` from every `as_*`** on a `Value` | the key holds a *string*, or is missing (`Null`) — dynamic access never panics, so the error is a `None` | `is_null()` for present-null vs absent, `.get()` for truly-missing; then `as_str().unwrap().parse()` |
| **`UnequalLengths`** from `csv::Reader` | a row has the wrong field count | fix the feed, or `flexible(true)` if ragged rows are the contract |
| **scraper returns 0 matches** on markup you can see | the html5 parser **dropped** the element (orphan `<tr>`) — *silently* | wrap in the grammar-required parent (`<table>`); §8.3 is the receipt |
| **`Regex::new` returns `Err`** | your *pattern* is invalid syntax | fix the pattern; it is checked once, at `Regex::new` |

The kernel-era traps from chapter 23 still stand and matter double here, because the polars compile is the longest thing this notebook does:

- **One crate cell per session.** Re-running §0.1 mid-session with a *changed* feature set forces a rebuild — minutes, not seconds. Decide the feature set once (§0) and leave it.
- **`{:?}` for library error types.** `e.classify()` (serde_json) and `e.kind()` (csv) return enums without `Display` impls; `{:?}` is the only format that works (§5.3, §3.3).


## 11. Mental Model: The Receipts

Every behavioural claim in this notebook was **probe-verified in this exact kernel** before it was taught (probe desks `p24a`/`p24b`; full logs beside the notebooks). The receipts, one per claim:

- **P1** named captures: `caps["sym"]`/`caps["side"]`/`caps["px"]`, `caps.len() == 4` (whole match is group 0).
- **P2** `replace_all` with `$side $sym @ $px` expansion; `RegexSet::matches` -> `[0, 1]` for a two-of-three line.
- **P3** regex `split` on `[,;]\s*` -> 3 fields; Python end-separator rule (empty items) preserved.
- **P4** inline flags `(?im)` classify 3 of 4 log lines, `c[2]` = message group.
- **P6** csv `HashMap` rows: 2 rows, `rows[1]["px"] == "418.12"`.
- **P8** csv writer: `sym,qty\nAAPL,100\n` — `\n` terminator on this toolchain (see the writer note in §12 before asserting on line endings).
- **P9** polars from CSV **in-kernel**: `shape = (2, 3)`, frame printed; `SerReader::finish` in scope; the compile is the multi-minute cost, cached per session.
- **Q1** `Value` dynamic access: `v["nope"].is_null() == true`; `as_f64` on `px` -> `Some(228.6)`.
- **Q2** `from_str` error on `tru`: `Syntax` class, `line 1 col 27`; the error's `Category` enum is `{:?}`-only.
- **Q3** hand-built `Map` -> `{"qty":100,"sym":"AAPL"}` (keys serialized sorted).
- **Q4** derive roundtrip text + `to_value`/`from_value` bridge both `true`.
- **Q5** a **persisted** derive struct (items level, across cells) roundtrips through `from_value`.
- **Q6** scraper `parse_document` + `tr.fill td` -> 4 cells; `a[href^="/fills/"]` -> 1.
- **Q7** **orphan `<tr>`: 0 matches, silently dropped** by html5ever — the loudest quiet failure in the chapter.
- **Q8** `p.price.up` -> `["228.60"]`; invalid selector `a >>` rejected by `Selector::parse`.
- **Q11** csv `deserialize()` in-kernel: quoted comma lands in one `String` field.


## 12. Cheat Sheet

```rust
// regex -------------------------------------------------------------
let re = Regex::new(r"(?<sym>[A-Z]{3,4})-(?<side>BUY|SELL)@(?<px>[0-9]+\.[0-9]{2})").unwrap();
re.is_match(hay);                       // bool
re.find(hay).unwrap().as_str();         // first match as &str
let c = re.captures(hay).unwrap();      // &c["sym"], &c["px"], c.len()
re.captures_iter(hay);                  // every match (finditer)
re.replace_all(hay, "$sym/$side");      // Cow<str>; .into_owned() to keep
re.split(hay);                          // iterator of &str
RegexSet::new([p1, p2])?.matches(hay);  // which patterns hit
// (?i) case, (?m) ^$ per line, (?s) dot-all — inline, per pattern

// csv ---------------------------------------------------------------
let mut r = csv::Reader::from_reader(data.as_bytes());
r.headers();                            // StringRecord (clone it to keep)
for rec in r.records() { rec.get(0); }  // StringRecord, by index
r.deserialize::<Fill>()                 // typed rows, per-row Result
ReaderBuilder::new().has_headers(false) // first row as data -> zip keys
csv::Writer::from_writer(vec![])        // .write_record([...]) -> RFC-4180
ReaderBuilder::new().flexible(true)     // ragged rows allowed

// serde_json --------------------------------------------------------
let v: Value = serde_json::from_str(s)?;      // dynamic, never panics on []
v["a"][0]["b"].as_f64();                      // Option; missing -> Null
e.classify(); e.line(); e.column();           // Syntax|Io|Data|Eof + coords
serde_json::to_string(&f)?; from_str::<Fill>? // derive both directions
serde_json::to_value(&f)?; from_value::<Fill> // Value bridge

// polars (from CSV, in memory) --------------------------------------
use polars::prelude::*;                        // + use std::io::Cursor;
let df = CsvReadOptions::default().with_has_header(true)
    .into_reader_with_file_handle(Cursor::new(text)).finish()?;
df.shape(); df.height(); df.dtypes();          // schema is part of the value
df.column("qty")?.as_series()?;                // by name; .i64() for the view
df.clone().lazy().filter(col("side").eq(lit("BUY")))
    .group_by([col("sym")]).agg([col("qty").sum()])
    .sort(["sym"], SortMultipleOptions::default()).collect()?;   // query plan

// scraper -----------------------------------------------------------
let doc = Html::parse_document(page);          // or parse_fragment(snippet)
let sel = Selector::parse("tr.fill td")?;      // CSS, checked once
doc.select(&sel);                              // iterate matching elements
el.text().collect::<String>();                 // text pieces joined
el.value().attr("href");                       // Option<&str>
// <tr> outside a table is silently dropped by the html5 grammar
```

**Writer/terminator note:** the csv writer emits `\n` line endings on this
toolchain — matching what the reader accepts. Do not assert for `\r\n`
without checking the builder's `terminator` setting first.


## 13. Review

**The mental model.** Text is ambiguous; parsers end the ambiguity at the door. Each crate in this chapter is a different door into the same bargain: declare the target (`Captures` names, a struct, a `Value`, a schema, a selector), parse once, get either a typed value or a classified error. After the boundary, the compiler owns the correctness — which is why the *choosing* (§9) is the real skill and the calls are muscle memory.

**Retrieval questions:**

1. Why is `caps["sym"]` safe but `caps["nope"]` a panic risk? *(What does the `Index` impl actually look up — and what is the `Option`-returning alternative?)*
2. What does `v["missing"]` return, and what distinguishes "absent" from "present but null"?
3. Two serde crate-instances in one process: which diagnostic appears, and on which side (derive or consumer)?
4. Where does `finish` live for `CsvReader`, and what was chapter 15's identical lesson?
5. A scraper query over visible `<tr>` markup returns nothing. Name the two-step diagnosis.
6. `replace_all` returns `Cow<str>`. When is it borrowed, and what must you do before binding it at cell top level?

**Answers:** (1) `Captures: Index<&str>` panics on an unknown *name* — prefer `c.get("sym")` (an `Option`) in code that does not own the pattern. (2) `Value::Null`; indexing cannot distinguish — `.get("missing")` returns `Option<&Value>` (`None` = absent, `Some(Null)` = present-null). (3) E0277 at the derive/consumer seam — impls from one instance are invisible to the other; one instance per process. (4) On the `SerReader` trait; ch15's `Datelike`/`Timelike` (trait methods need their trait in scope). (5) Is the selector compiled (`Selector::parse` ok)? Then is the element in a grammar-valid context — an orphan `<tr>` is dropped. (6) Borrowed when nothing matched; `.into_owned()`.

**What comes next.** Chapter 25 is the one the series has been building credibility for: **unit tests** — `#[test]`, the assertion family, fixtures and the Python `unittest` mindset translated into Rust's built-in harness. Chapter 26 closes the DLC with **macros** — the `vec!`/`println!` machinery you have been *using* since chapter 1, finally written by hand.
